# Computational X-Ray Diffraction Analysis of Cotton Fibre

- **Notebook:** `notebooks/04_xrd_analysis.ipynb`
- **Author:** Shakib Hasan
- **Status:** Day 1 scaffold. Question, source notes, concepts and limits only; no data are loaded yet.
- **Data type:** Published, open-access experimental data. Analysed by me; not measured by me.

## 1. Project question

What can one published cotton-fibre X-ray diffraction (XRD) pattern reveal about:

1. **Peak positions:** where do the clearly visible diffraction features appear on the 2θ axis?
2. **d-spacing:** what interplanar spacings do those positions correspond to, through Bragg's law?
3. **An empirical crystallinity index:** what does the Segal method give, and how sensitive is it to the choices I make?
4. **Apparent coherent-domain size:** can a defensible Scherrer estimate be made from one well-resolved peak, and what are its limits?

**Approach:** published data, transparent calculations, cautious interpretation and reproducible Python code.

**Not the goal:** to prove a crystal structure, to measure the "true" crystalline fraction, or to explain why this cotton behaves the way it does.

## 2. Source notes

*Copied from `data/README.md` (written in Block A). If this cell and `data/README.md` ever disagree, the original file and `data/README.md` win; fix this cell.*

- **Article:** Nam, S., French, A. D., & Kashem, M. N. H. (2026). *Open-source Rietveld refinement for cellulose crystallinity using a laboratory powder X-ray diffractometer: a hands-on tutorial.* Cellulose.
- **DOI:** https://doi.org/10.1007/s10570-026-07136-7
- **File used:** Supplementary file 1 (cotton XRD pattern), exact file name as downloaded: `10570_2026_7136_MOESM1_ESM.txt`
- **Downloaded on:** 2026-10-12
- **Raw-file storage decision:** Raw file not committed; download instructions in `data/README.md`
- **File structure, from manual inspection in a text editor (Block A):** 813 rows; two tab-separated columns; no header; first column runs from 9.88213312 to 41.86750172 in steps of about 0.03939; second column holds integer values from 6539 to 47186, with the largest value near 22.68 in the first column.
- **Column meaning:** column 1 is assumed to be the angle axis and column 2 the intensity. The units of both are **not stated in the file** and will be confirmed against the article on Day 2.

### 2.1 Open questions (not resolved yet; nothing assumed)

- **Scan range:** the file covers about 9.88° to 41.87°, but the article reports a scan range of 5–80°. I do not yet know why they differ. To be checked against the article and its supplementary information on Day 2.
- **Radiation / wavelength:** not encoded in the file. It will be taken from the article and labelled "article-reported" on Day 2.
- **Intensity units:** not stated in the file. To be confirmed on Day 2.
- **Date / log:** 2026-10-14: scaffold written; no data analysed yet.

## 3. Crystalline versus amorphous order

**Crystalline order** means the repeating units (cellulose chains in cottons) sit in a regular pattern that repeats in all three directions. Sets of parallel planes in that pattern scatter X-rays in step with one another at particular angles, which produces **relatively sharp diffraction peaks**.

**Amorphous (disordered) material** has no long-range repeat, so scattering from different parts of the sample does not add up at specific angles. It appears as a **broad, diffuse hump** spread over a wide angular range instead of narrow peaks.

**Why cellulose shows both:** native cotton cellulose is **semicrystalline**. Ordered cellulose crystallites sit alongside less ordered regions (disordered cellulose, chain-surface and defect regions, and other minor components). The detector records the *sum* of everything, so one pattern can contain peaks from the ordered parts on top of a broad diffuse contribution.

**Two cautions I will carry through this project:**
- Cellulose crystallites are very small, so even the "crystalline" peaks are broad and neighbouring peaks overlap. This is one reason a clean split into "crystalline" and "amorphous" parts is difficult.
- Diffuse intensity does not come only from amorphous cellulose. Background from the instrument and sample holder, and other scattering, can also contribute.


## 4. XRD foundations: Bragg's law, wavelength and d-spacing

### 4.1 Bragg's law

$$2\,d\,\sin\theta = n\,\lambda$$

| Symbol | Meaning | Unit used here |
|---|---|---|
| $\lambda$ | wavelength of the X-ray beam | ångström (Å); 1 Å = 0.1 nm |
| $d$ | interplanar spacing: the perpendicular distance between neighbouring planes in one family of parallel lattice planes | Å (same unit as $\lambda$) |
| $\theta$ | Bragg angle: the angle between the incoming beam and the lattice planes | degrees (converted to radians in code) |
| $n$ | order of reflection (a whole number); working assumption here: $n = 1$ | none |

**Idea in words:** X-rays scattered from two adjacent planes travel different distances. The extra path is $2d\sin\theta$. The two waves recombine constructively only if that extra path is a whole number of wavelengths. That condition produces a diffraction peak.

**Wavelength:** $\lambda$ is a property of the X-ray source. For this dataset its value must come from the article; I will not assume it.

**d-spacing:** $d$ is a distance between planes inside the crystal structure. It is not the size of a crystallite or of a fibre.

### 4.2 What 2θ is, and why Bragg's law uses θ

- A powder diffractometer reports **2θ**: the angle between the original beam direction and the direction of the diffracted beam.
- Planes tilted at θ to the incoming beam reflect it so that it leaves at θ on the other side. In total the beam is turned through **2θ**.
- Bragg's law is written in terms of **θ**, so the instrument angle must be halved first: $\theta = (2\theta)/2$.
- A common error is putting the 2θ value straight into the sine. This is checked explicitly on Day 3.
- Whether column 1 of this file is a 2θ axis will be confirmed against the article on Day 2. The convention above is the standard one for a powder diffractometer.
- At fixed wavelength, a larger θ means a smaller $d$. The worked explanation and calculations come on Day 3.

## 5. Boundaries of this analysis (what this notebook does not claim)

1. **Published data, not my experiment.** I am analysing an open, published dataset. I did not prepare the sample, or collect the pattern. Any description of the measurement is article-reported.
2. **One pattern cannot prove cause.** A single diffraction pattern from one sample cannot show that any processing or treatment caused a change in a property. That would need controlled comparisons and replicates.
3. **Scherrer size is not fibre diameter.** A Scherrer estimate describes an apparent coherent-domain size: the extent over which lattice planes diffract coherently, perpendicular to the reflecting planes. Peak broadening can also come from the instrument and from microstrain. A whole cotton fibre is micrometre-scale. The two quantities are different and must not be compared as if they were the same thing.

*Further limits (for example, that the Segal index is empirical and that d-spacing alone does not identify a material) will be added when each method is introduced.*

## Load and validate the raw diffraction data
Goal: turn the downloaded source file into a validated table, without changing any values.

In [1]:
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from pathlib import Path

print("pandas", pd.__version__)
print("numpy", np.__version__)
print("matplotlib", matplotlib.__version__)

pandas 3.0.6
numpy 2.5.3
matplotlib 3.11.2


In [2]:
PROJECT_ROOT = Path.cwd().parent        # notebooks/ -> repo root
RAW_DIR = PROJECT_ROOT / "data" / "raw"

print("Working folder:", Path.cwd())
print("Files in data/raw:", sorted(p.name for p in RAW_DIR.glob("*")))

Working folder: c:\Users\User\Documents\GitHub\fiber-characterization-portfolio\notebooks
Files in data/raw: ['10570_2026_7136_MOESM1_ESM.txt']


In [3]:
DATA_FILE = RAW_DIR / "10570_2026_7136_MOESM1_ESM.txt"
print("File exists:", DATA_FILE.exists())

File exists: True


In [4]:
with open(DATA_FILE, "r", encoding="utf-8") as f:
    all_lines = f.read().splitlines()

print("Total lines:", len(all_lines))
print("Blank lines:", sum(1 for line in all_lines if line.strip() == ""))
print("Lines with exactly one tab:", sum(1 for line in all_lines if line.count("\t") == 1))
print()
for i, line in enumerate(all_lines[:5], start=1):
    print(i, repr(line))
print("...")
print(len(all_lines), repr(all_lines[-1]))

Total lines: 813
Blank lines: 0
Lines with exactly one tab: 813

1 '9.88213312\t6539'
2 '9.92152397\t6739'
3 '9.96091482\t6593'
4 '10.00030566\t6585'
5 '10.03969651\t6681'
...
813 '41.86750172\t7661'


In [5]:
raw = pd.read_csv(DATA_FILE, sep="\t", header=None)
print("Shape (rows, columns):", raw.shape)
raw.head()

Shape (rows, columns): (813, 2)


,0,1
0,9.882133,6539
1,9.921524,6739
2,9.960915,6593
3,10.000306,6585
4,10.039697,6681


In [6]:
df = raw.copy()
df.columns = ["two_theta_deg", "intensity"]

In [7]:
display(df.head(10))
display(df.tail(5))
print("Number of observations:", len(df))
print()
df.info()
display(df.describe().T)
print()
print(f"2-theta min/max : {df['two_theta_deg'].min():.8f} / {df['two_theta_deg'].max():.8f}")
print(f"Intensity min/max: {df['intensity'].min()} / {df['intensity'].max()}")

,two_theta_deg,intensity
0,9.882133,6539
1,9.921524,6739
2,9.960915,6593
3,10.000306,6585
4,10.039697,6681
5,10.079087,6702
6,10.118478,6741
7,10.157869,6758
8,10.197260,6932
9,10.236651,6753


,two_theta_deg,intensity
808,41.709938,7660
809,41.749329,7481
810,41.788720,7622
811,41.828111,7618
812,41.867502,7661


Number of observations: 813

<class 'pandas.DataFrame'>
RangeIndex: 813 entries, 0 to 812
Data columns (total 2 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   two_theta_deg  813 non-null    float64
 1   intensity      813 non-null    int64  
dtypes: float64(1), int64(1)
memory usage: 12.8 KB


,count,mean,std,min,25%,50%,75%,max
two_theta_deg,813.0,25.874817,9.250436,9.882133,17.878475,25.874817,33.87116,41.867502
intensity,813.0,11823.297663,7464.992571,6539.000000,7812.000000,8582.000000,12741.00000,47186.000000



2-theta min/max : 9.88213312 / 41.86750172
Intensity min/max: 6539 / 47186


In [8]:
print("Missing values per column:")
print(df.isna().sum())
print()
print("dtypes:")
print(df.dtypes)
print()
print("All angles finite   :", np.isfinite(df["two_theta_deg"]).all())
print("All intensities finite:", np.isfinite(df["intensity"]).all())

Missing values per column:
two_theta_deg    0
intensity        0
dtype: int64

dtypes:
two_theta_deg    float64
intensity          int64
dtype: object

All angles finite   : True
All intensities finite: True


In [9]:
angle = df["two_theta_deg"]
print("Duplicate angle values:", angle.duplicated().sum())
print("Fully duplicated rows :", df.duplicated().sum())
print("Strictly increasing   :", angle.is_monotonic_increasing and angle.is_unique)

Duplicate angle values: 0
Fully duplicated rows : 0
Strictly increasing   : True


In [10]:
step = angle.diff().dropna()
print("Number of steps:", len(step))
print(f"Mean   step: {step.mean():.8f} deg")
print(f"Median step: {step.median():.8f} deg")
print(f"Min    step: {step.min():.8f} deg")
print(f"Max    step: {step.max():.8f} deg")
print(f"Largest deviation from median: {(step - step.median()).abs().max():.2e} deg")

expected_rows = (angle.max() - angle.min()) / step.median() + 1
print(f"Rows expected from range/step: {expected_rows:.2f}  | rows present: {len(df)}")

Number of steps: 812
Mean   step: 0.03939085 deg
Median step: 0.03939085 deg
Min    step: 0.03939084 deg
Max    step: 0.03939085 deg
Largest deviation from median: 1.00e-08 deg
Rows expected from range/step: 813.00  | rows present: 813


In [11]:
intensity = df["intensity"]
print("All values integer-valued:", bool((intensity == intensity.round()).all()))
print("Any zero or negative     :", bool((intensity <= 0).any()))
print("Row with the largest intensity:")
print(df.loc[intensity.idxmax()])

All values integer-valued: True
Any zero or negative     : False
Row with the largest intensity:
two_theta_deg       22.684159
intensity        47186.000000
Name: 325, dtype: float64


### Data-quality note

**File:** `10570_2026_7136_MOESM1_ESM.txt`, stored locally in `data/raw/` (not committed).

**Structure (from the file itself):** 813 rows, two tab-separated columns, no header.

**Column meaning:** The names `two_theta_deg` and `intensity` are **inferred, not stated by the authors** (see "Supplementary file 1" below). The basis is that column 1 increases monotonically from 9.88213312 to 41.86750172 (consistent with 2θ in degrees) and column 2 holds positive integers (consistent with detector counts).

| Check | Result |
|---|---|
| Missing / non-numeric / non-finite | 0 |
| Duplicate angles | 0 |
| Strictly increasing angles | True |
| Step size (median, largest deviation) | 0.03939085, 1.00e-08 deg |
| Gaps (expected rows vs present) | 0 |
| Intensity range, integer-valued | 6539 to 47186; all 813 values integer-valued (int64); none zero or negative |

**Observed scan range (file):** 9.88213312 to 41.86750172 degrees 2θ.

**Article-reported measurement details (not encoded in the file):**
- Radiation / wavelength: Cu Kα with a nickel filter; average wavelength 1.5418 Å (Materials and methods → XRD experiment).
- Reported scan range: 5–80° 2θ (Materials and methods → XRD experiment).
- Step size / scan speed: step size 0.039° (Materials and methods → XRD experiment); scan speed not stated.
- Instrument: Empyrean diffractometer, Malvern Panalytical (Materials and methods → XRD experiment).
- Sample: cotton fibers from plants grown at the Southern Regional Research Center, New Orleans (Materials and methods → Materials); mechanically ground in a Wiley cutting mill with a 40-mesh screen (Materials and methods → Sample preparation).
- Supplementary file 1: no separate caption. The Supplementary information item describes it as a plain text file of the cotton fiber XRD data, formatted for import into MAUD. The main text (Results and discussion → Main inputs to MAUD → Load data file) calls it "cotton.txt" and describes it as the saved XRD pattern. Neither says what the two columns are.

**Comparison:**
- Step size: the file's median step is [median step from Step 9] °, consistent with the reported 0.039° (step reported to three decimals).
- Scan range: the file covers 9.88213312 to 41.86750172 ° 2θ, which is narrower than the reported 5–80°.
- Filename: the downloaded file is `10570_2026_7136_MOESM1_ESM.txt`, while the main text calls it "cotton.txt". The difference is likely from the publisher's renaming, but the article does not say so.

**Unresolved:**
- Column meaning and intensity units are not stated in the article or supplement.
- The reason the file's range is narrower than the reported scan range is not stated (trimmed for MAUD import, or otherwise).
- Scan speed is not stated.